# Inference
Loads `weights_v2/latest.pt` from Drive and translates. Runtime must be **GPU** (or CPU, just slower).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT = '/content/drive/MyDrive/en_ar_transformer'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip -q install datasets tokenizers tensorboard sacrebleu altair

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.1 MB/s eta 0:00:00


In [2]:
import torch
from config import get_config
from train import load_data
from decoding import Translator, show_examples, translate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
translator = Translator(config, device)          # loads model + tokenizers + latest checkpoint

Using device: cuda
Loaded /content/drive/MyDrive/en_ar_transformer/weights_v2/latest.pt  (trained epochs: 12)


## Validation examples (greedy vs beam)

In [3]:
_, val_dl, _, _, _ = load_data(config)           # val_dl has batch_size=1
show_examples(translator, val_dl, n=10, beam_size=4)

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

ar-en/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  214kB            

ar-en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ar-en/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 99.3MB            

ar-en/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/300000 [00:00<?, ? examples/s]

Kept 261190/261193 pairs after token-length filter
Kept 29022/29022 pairs after token-length filter
--------------------------------------------------------------------------------
             SOURCE: Haven't you ever heard that the unexamined life is not worth living?
             TARGET: ما أنت أبدا سمعت ذلك الحياة الغير مفحوصة ألن يساوي يعيش؟
             GREEDY: ألم تسمع ذلك من قبل حياة لا يستحق الحياة التي لا يستحق أن تكون على الإطلاق؟
               BEAM: ألم تسمع أن الحياة التي لم أسمع بها؟
--------------------------------------------------------------------------------
             SOURCE: It's clonefeed!
             TARGET: إنه الاستنساخ الغذائي
             GREEDY: إنه أمر محض
               BEAM: ! إنه أمر جيد
--------------------------------------------------------------------------------
             SOURCE: Statement submitted by Equality Now, a non-governmental organization in consultative status with the Economic and Social Council
             TARGET: بيان مقدم من من

## Translate your own sentences

In [4]:
for s in ["Why do I need to translate this?",
          "Where is the nearest hospital?",
          "I will see you tomorrow.",
          "The committee adopted the report without a vote."]:
    print(s)
    print("  greedy:", translator.translate(s, method="greedy"))
    print("  beam  :", translator.translate(s, method="beam", beam_size=4))

Why do I need to translate this?
  greedy: لماذا أحتاج إلى هذا العصا؟
  beam  : لماذا يجب أن أفعل هذا؟
Where is the nearest hospital?
  greedy: أين هو المستشفى؟
  beam  : أين هو المستشفى؟
I will see you tomorrow.
  greedy: سوف أراك غدا
  beam  : سوف أراك غدا
The committee adopted the report without a vote.
  greedy: اعتمدت اللجنة الفرعية دون تصويت.
  beam  : اعتمدت اللجنة الفرعية دون تصويت.


In [5]:
# one-liner version (loads the model on first call)
translate("How are you doing today?")

Loaded /content/drive/MyDrive/en_ar_transformer/weights_v2/latest.pt  (trained epochs: 12)


'كيف حالك اليوم؟'